# EDA (Exploratory Data Analysis) of the Abalone dataset

**Goal of the project:** predict the age of an abalone from physical measurements, without cutting and staining the shell.

**Target:** `Rings`. By convention, age in years ≈ `Rings + 1.5`, so predicting `Rings` is the same as predicting age.

**Features:**

| Column | Type | Unit | Description |
| --- | --- | --- | --- |
| `Sex` | categorical | | M (male), F (female), I (infant) |
| `Length` | continuous | mm / 200 | Longest shell measurement |
| `Diameter` | continuous | mm / 200 | Perpendicular to length |
| `Height` | continuous | mm / 200 | With meat in shell |
| `Whole weight` | continuous | g / 200 | Whole abalone |
| `Shucked weight` | continuous | g / 200 | Weight of meat |
| `Viscera weight` | continuous | g / 200 | Gut weight (after bleeding) |
| `Shell weight` | continuous | g / 200 | After being dried |
| `Rings` | integer | | Target |

The continuous values were divided by 200 in the original dataset.

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 500)

# Recessive axes and grid, so the data stands out
plt.rcParams.update(
    {
        "figure.dpi": 80,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#8a8984",
        "axes.labelcolor": "#52514e",
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.color": "#e6e5e0",
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "legend.frameon": False,
    }
)

# Fixed color per sex (first three slots of a colorblind-safe categorical palette)
SEX_ORDER = ["F", "M", "I"]
SEX_COLORS = {"F": "#2a78d6", "M": "#eb6834", "I": "#1baf7a"}
SEX_LABELS = {"F": "Female", "M": "Male", "I": "Infant"}
MAIN_COLOR = "#2a78d6"

# Data

In [ ]:
df = pd.read_csv("../data/abalone.csv")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df.dtypes

In [ ]:
TARGET = "Rings"
NUM_FEATURES = [
    "Length",
    "Diameter",
    "Height",
    "Whole weight",
    "Shucked weight",
    "Viscera weight",
    "Shell weight",
]
CAT_FEATURES = ["Sex"]

# EDA

## 1. Data quality

We check for missing values, duplicates and values that are physically impossible.

In [ ]:
df.describe().T

In [ ]:
quality = pd.DataFrame(
    {
        "n_missing": df.isna().sum(),
        "n_zero": (df == 0).sum(),
        "n_unique": df.nunique(),
    }
)
print(f"Duplicated rows: {df.duplicated().sum()}")
quality

There are no missing values and no duplicates. But `Height` has **2 rows equal to 0**, which is impossible for a real animal.

We also check that the weights are consistent with each other:
- the meat (`Shucked weight`) cannot be heavier than the whole animal,
- the sum of the parts should be about equal to `Whole weight` or smaller (some water and blood are lost when the animal is shucked).

In [ ]:
parts_sum = df["Shucked weight"] + df["Viscera weight"] + df["Shell weight"]

checks = {
    "Height == 0": df["Height"] == 0,
    "Shucked weight > Whole weight": df["Shucked weight"] > df["Whole weight"],
    "Viscera weight > Whole weight": df["Viscera weight"] > df["Whole weight"],
    "Shell weight > Whole weight": df["Shell weight"] > df["Whole weight"],
    "Sum of parts > Whole weight": parts_sum > df["Whole weight"] + 1e-9,
    "Sum of parts > 1.5 x Whole weight": parts_sum > 1.5 * df["Whole weight"],
}
pd.Series({name: mask.sum() for name, mask in checks.items()}, name="n_rows")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ratio_bad = checks["Sum of parts > 1.5 x Whole weight"]
ax.scatter(
    df.loc[~ratio_bad, "Whole weight"],
    parts_sum[~ratio_bad],
    s=6,
    alpha=0.35,
    color=MAIN_COLOR,
    linewidths=0,
    label="Consistent rows",
)
ax.scatter(
    df.loc[ratio_bad, "Whole weight"],
    parts_sum[ratio_bad],
    s=40,
    facecolors="none",
    edgecolors="#0b0b0b",
    linewidths=1.2,
    label="Sum of parts > 1.5 x whole",
)
lim = [0, 3]
ax.plot(lim, lim, color="#8a8984", linewidth=1, linestyle="--")
ax.annotate(
    "parts = whole",
    (2.3, 2.3),
    xytext=(-75, 8),
    textcoords="offset points",
    fontsize=8,
    color="#52514e",
)
ax.set_xlabel("Whole weight")
ax.set_ylabel("Shucked + Viscera + Shell weight")
ax.set_title("Sum of the parts vs whole weight")
ax.legend(loc="upper left")
plt.show()

df[checks["Height == 0"] | ratio_bad]

- 2 rows have `Height == 0` (both infants). This is probably a missing value saved as 0.
- 4 rows have `Shucked weight > Whole weight`. This is impossible, so it is a data entry error.
- 153 rows (~3.7%) have a sum of parts larger than `Whole weight`. For most of them the gap is small (median ≈ 4%), which is plausible measurement noise: the parts are weighed separately, and the shell after drying. We keep these rows.
- But 7 rows (all infants) have parts that weigh **more than 1.5 times** the whole animal, up to 5 times. These include the 4 rows above and are clearly wrong.

### Outliers in `Height`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

sns.boxplot(x=df["Height"], ax=axes[0], color=MAIN_COLOR, fliersize=3, linewidth=1)
axes[0].set_title("Height distribution")

axes[1].scatter(
    df["Length"], df["Height"], s=6, alpha=0.35, color=MAIN_COLOR, linewidths=0
)
extreme = df[df["Height"] > 0.4]
axes[1].scatter(
    extreme["Length"],
    extreme["Height"],
    s=40,
    facecolors="none",
    edgecolors="#0b0b0b",
    linewidths=1.2,
)
for idx, row in extreme.iterrows():
    axes[1].annotate(
        f"row {idx}: Height={row['Height']}",
        (row["Length"], row["Height"]),
        xytext=(-130, -4),
        textcoords="offset points",
        fontsize=8,
        color="#52514e",
    )
axes[1].set_xlabel("Length")
axes[1].set_ylabel("Height")
axes[1].set_title("Height vs Length")

plt.tight_layout()
plt.show()

extreme

Two rows have a height far from all other abalones (1.13 and 0.515, while the 99th percentile is around 0.22). Their length and weight are normal, so the height is a typo (for example 0.113 written as 1.13). They are clear outliers.

**Cleaning decision for the training code:** drop the rows with `Height == 0`, `Height > 0.4`, or a sum of parts above 1.5 × `Whole weight` (10 rows, 0.2% of the data).

In [ ]:
invalid = (
    (df["Height"] == 0) | (df["Height"] > 0.4) | (parts_sum > 1.5 * df["Whole weight"])
)
df_clean = df[~invalid].reset_index(drop=True)
print(f"Removed {invalid.sum()} rows -> {len(df_clean)} rows left")

The rest of the analysis uses `df_clean`.

## 2. Target: `Rings`

In [ ]:
rings = df_clean[TARGET]

fig, ax = plt.subplots(figsize=(10, 3.8))
counts = rings.value_counts().sort_index()
ax.bar(counts.index, counts.values, width=0.8, color=MAIN_COLOR)
ax.axvline(rings.mean(), color="#0b0b0b", linewidth=1, linestyle="--")
ax.annotate(
    f"mean = {rings.mean():.1f}",
    (rings.mean(), counts.max()),
    xytext=(6, -10),
    textcoords="offset points",
    fontsize=9,
    color="#0b0b0b",
)
ax.set_xlabel("Rings")
ax.set_ylabel("Number of abalones")
ax.set_title("Distribution of the target (Rings)")
ax.set_xticks(range(1, rings.max() + 1, 2))
plt.show()

print(
    f"Mean: {rings.mean():.2f} | Median: {rings.median():.0f} | Std: {rings.std():.2f} | Skew: {rings.skew():.2f}"
)
print(f"Range: {rings.min()} to {rings.max()}")
print(f"Share of abalones with 7 to 13 rings: {rings.between(7, 13).mean():.1%}")
print(f"Share of abalones with more than 20 rings: {(rings > 20).mean():.1%}")

- The target is an integer between 1 and 29, mostly between 7 and 13 rings.
- It is **right-skewed** (skew ≈ 1.1): there is a long tail of old abalones with few samples. Some classes have only 1 or 2 examples.
- So it is better to treat the problem as a **regression**, not a classification with 28 classes. The model will probably make larger errors on old abalones, because there are few examples of them.
- A log transform of the target (`log1p(Rings)`) can be tested to reduce the skew.

## 3. Categorical feature: `Sex`

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.6]}
)

sex_counts = df_clean["Sex"].value_counts().reindex(SEX_ORDER)
bars = axes[0].bar(
    [SEX_LABELS[s] for s in SEX_ORDER],
    sex_counts.values,
    color=[SEX_COLORS[s] for s in SEX_ORDER],
    width=0.6,
)
axes[0].bar_label(bars, padding=3, fontsize=9, color="#0b0b0b")
axes[0].set_ylabel("Number of abalones")
axes[0].set_title("Count by sex")

sns.boxplot(
    data=df_clean,
    x="Sex",
    y=TARGET,
    order=SEX_ORDER,
    hue="Sex",
    hue_order=SEX_ORDER,
    palette=SEX_COLORS,
    ax=axes[1],
    fliersize=2,
    linewidth=1,
    legend=False,
)
axes[1].set_xticks(range(3), [SEX_LABELS[s] for s in SEX_ORDER])
axes[1].set_xlabel("")
axes[1].set_title("Rings by sex")

plt.tight_layout()
plt.show()

df_clean.groupby("Sex")[TARGET].describe().reindex(SEX_ORDER).round(2)

- The 3 classes are balanced (about 1/3 each).
- **Infants are clearly younger** (median 8 rings) than adults (median 10 rings). Males and females have almost the same distribution.
- `Sex` is useful mostly as "infant vs adult". We will one-hot encode it.

## 4. Numerical features

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, NUM_FEATURES):
    ax.hist(df_clean[col], bins=40, color=MAIN_COLOR, histtype="stepfilled")
    ax.set_title(col)
    ax.set_ylabel("")
axes.flat[-1].axis("off")
fig.suptitle("Distribution of the numerical features", fontweight="bold")
plt.tight_layout()
plt.show()

df_clean[NUM_FEATURES].skew().round(2).rename("skew").to_frame().T

- The size features (`Length`, `Diameter`, `Height`) are slightly left-skewed.
- The weights are right-skewed. This makes sense: weight grows like the volume, so roughly like length³.
- All features are on similar small scales (0 to 3), but scaling is still useful for linear models.

### Weight vs size: a non-linear relation

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for sex in SEX_ORDER:
    sub = df_clean[df_clean["Sex"] == sex]
    ax.scatter(
        sub["Length"],
        sub["Whole weight"],
        s=6,
        alpha=0.4,
        color=SEX_COLORS[sex],
        label=SEX_LABELS[sex],
        linewidths=0,
    )
ax.set_xlabel("Length")
ax.set_ylabel("Whole weight")
ax.set_title("Whole weight grows like a power of Length")
ax.legend(markerscale=3)
plt.show()

Weight is a convex function of length (close to a cubic). A linear model will not capture this alone, so tree-based models, or polynomial/log features, should work better.

## 5. Features vs target

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6), sharey=True)
for ax, col in zip(axes.flat, NUM_FEATURES):
    for sex in SEX_ORDER:
        sub = df_clean[df_clean["Sex"] == sex]
        ax.scatter(
            sub[col],
            sub[TARGET],
            s=4,
            alpha=0.3,
            color=SEX_COLORS[sex],
            label=SEX_LABELS[sex],
            linewidths=0,
        )
    ax.set_title(col)
axes[0, 0].set_ylabel(TARGET)
axes[1, 0].set_ylabel(TARGET)
axes.flat[-1].axis("off")
handles, labels = axes[0, 0].get_legend_handles_labels()
axes.flat[-1].legend(handles, labels, loc="center", markerscale=4, title="Sex")
fig.suptitle("Rings vs each numerical feature", fontweight="bold")
plt.tight_layout()
plt.show()

- All features have a **positive** relation with `Rings`: bigger and heavier abalones are older.
- The relation is **not linear**: it is steep for small abalones, then becomes flat. After a certain size, abalones keep adding rings but do not grow much.
- The **spread of `Rings` grows with size** (heteroscedasticity). For a small abalone, the age is easy to predict. For a large one, the age can be anything from 8 to 25 rings.
- Infants are concentrated on the bottom left, which confirms what we saw with the boxplot.

In [ ]:
size_bins = pd.qcut(df_clean["Shell weight"], q=10)
spread = df_clean.groupby(size_bins, observed=True)[TARGET].agg(["mean", "std"])

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(len(spread))
ax.plot(
    x,
    spread["mean"],
    color=MAIN_COLOR,
    linewidth=2,
    marker="o",
    markersize=5,
    label="Mean Rings",
)
ax.fill_between(
    x,
    spread["mean"] - spread["std"],
    spread["mean"] + spread["std"],
    color=MAIN_COLOR,
    alpha=0.15,
    linewidth=0,
    label="± 1 std",
)
ax.set_xticks(x, [f"D{i + 1}" for i in x])
ax.set_xlabel("Shell weight decile (D1 = lightest)")
ax.set_ylabel(TARGET)
ax.set_title("Mean and spread of Rings by Shell weight decile")
ax.legend(loc="upper left")
plt.show()

spread.round(2)

The mean number of rings keeps increasing with shell weight, but the standard deviation of `Rings` more than doubles from the lightest to the heaviest decile (≈ 1.6 to ≈ 3.6 rings). So the error of any model will be larger for big abalones.

## 6. Correlations

In [ ]:
corr = df_clean[NUM_FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(7.5, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=2,
    linecolor="white",
    cbar_kws={"shrink": 0.7, "label": "Pearson correlation"},
    annot_kws={"fontsize": 8},
    ax=ax,
)
ax.set_title("Correlation matrix (Pearson)")
ax.grid(False)
plt.show()

In [ ]:
target_corr = pd.DataFrame(
    {
        "pearson": df_clean[NUM_FEATURES].corrwith(df_clean[TARGET]),
        # Spearman = Pearson on the ranks
        "spearman": df_clean[NUM_FEATURES].rank().corrwith(df_clean[TARGET].rank()),
    }
).sort_values("spearman", ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(target_corr))
ax.barh(
    y + 0.2,
    target_corr["pearson"],
    height=0.38,
    color=MAIN_COLOR,
    label="Pearson (linear)",
)
ax.barh(
    y - 0.2,
    target_corr["spearman"],
    height=0.38,
    color="#eb6834",
    label="Spearman (monotonic)",
)
ax.set_yticks(y, target_corr.index)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel(f"Correlation with {TARGET}")
ax.set_title("Correlation of each feature with the target")
ax.legend(loc="lower right")
plt.show()

target_corr.round(3)

- **`Shell weight`** is the feature most correlated with `Rings` (Pearson ≈ 0.63, Spearman ≈ 0.69). `Shucked weight` is the least correlated.
- Spearman is higher than Pearson for every feature: the relations are monotonic but not linear, as we saw in the scatter plots.
- The features are **very correlated with each other** (0.84 to 0.99). `Length` and `Diameter` are almost the same (0.99). This multicollinearity is a problem for interpreting the coefficients of a linear model (use regularization like Ridge), but it is not a problem for tree-based models.

# Conclusions

**Data quality**
- 4,177 rows, 8 features, no missing values, no duplicates.
- 10 invalid rows (0.2%): `Height == 0` (2), `Height > 0.4` (2), sum of parts > 1.5 × `Whole weight` (7, one overlaps with `Height == 0`). We drop them in the training pipeline.

**Target**
- `Rings` is an integer from 1 to 29, centered on 9–10 and right-skewed, with very few old abalones. We treat it as a **regression** problem. A log transform of the target can be tested.

**Features**
- `Sex` mostly separates infants (younger) from adults; M and F are almost the same. One-hot encode it.
- All numerical features are positively and **non-linearly** related to `Rings`. `Shell weight` is the strongest predictor.
- The features are highly correlated with each other (multicollinearity).
- The variance of `Rings` grows with the size of the abalone: large abalones are hard to age from measurements only.

**Implications for modelling**
- Preprocessing: drop invalid rows, one-hot encode `Sex`, scale the numerical features (for linear models).
- Start with a regularized linear regression (Ridge) as a simple model, then try a tree-based model (Random Forest / Gradient Boosting), which handles non-linearity and correlated features better.
- Expect larger errors on old abalones (few samples and high variance).